# WAFER-GUARD · Fab-7 Excursion Detector — Pre-Release Model Audit

**Role:** you are the final technical reviewer on the release board.

**Status:** the model below clears the release gate (**F1 ≥ 0.90**). Deployment to the line-monitoring service is scheduled for Friday 06:00.

**Time-box:** 120 minutes.

## Your mandate

A staff engineer has a bad feeling about this notebook and has asked you to sign off — or to stop the release.

1. **Audit.** Identify *every* defect that invalidates the reported metric or would degrade the deployed model. For each defect give: (a) exact location, (b) mechanism, (c) **reproducible evidence from your own runs** — numbers, shapes, ablations. Assertions without evidence earn no credit. You are not told how many defects exist.
2. **Repair.** Deliver a corrected end-to-end pipeline that respects production causality: *no information from time *t′ > t* (or from the evaluation population) may influence a prediction made at time *t*.*
3. **Re-evaluate.** Report honest metrics at the **natural defect prevalence (~6.6 %)**. Justify your metric(s) and your decision-threshold procedure.
4. **Write-up.** ≤ 1 page: findings ranked by severity, residual risks, and your go / no-go recommendation.

> Deliverables: a repaired notebook + `AUDIT.md`. The existing notebook runs top-to-bottom without error. That is not evidence that it is correct.

### Data dictionary — `secom_telemetry_modified.csv`

| column | description |
|---|---|
| `sensor_0 … sensor_589` | 590 in-line process / metrology signals per wafer lot (anonymised) |
| `label` | electrical-test outcome: `-1` = pass, `+1` = fail |
| `timestamp` | lot start time (the fab runs continuously; equipment ages, recipes drift) |

**Fab tool-matching note (from the process-integration team):** a fraction of wafers are routed through *Probe Station PS-L (legacy)*, whose firmware emits telemetry in a different format than the current-generation stations. Consult the data.


## 0 · Environment

In [ ]:
!pip install "torch>=2.2" "scikit-learn>=1.2" "imbalanced-learn>=0.11" pandas numpy joblib matplotlib

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score, confusion_matrix)
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HP = dict(batch_size=64, epochs=40, lr=1e-3, weight_decay=1e-5,
          hidden=256, latent=32, lambda_clf=1.0, test_size=0.20)
print(f"torch {torch.__version__} | device={DEVICE}")

torch 2.3.1+cpu | device=cpu


## 1. Data


In [ ]:
df = pd.read_csv("secom_telemetry_modified.csv")

# FIX 1: Parse timestamp and sort chronologically
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)     #drop old index labels

SENSOR_COLS = [c for c in df.columns if c.startswith("sensor_")]
X_raw = df[SENSOR_COLS].to_numpy(dtype=np.float64)          # [N, 590] , numpy array of 64 bit floating point integer
y_raw = (df["label"].to_numpy() == 1).astype(np.int64)      # 1 = defective , numpy array of 64 bit integer

print(f"X: {X_raw.shape} | defect rate: {y_raw.mean():.2%} "
      f"({y_raw.sum()} fail / {(1 - y_raw).sum()} pass)")
print(f"missing cells: {np.isnan(X_raw).mean():.2%}")
print(f"timestamp range: {df['timestamp'].min()} to {df['timestamp'].max()}")

# FIX 3: Identify legacy probe station rows (PS-L)
legacy_mask = (df["sensor_0"] == 0.0) & (df["sensor_1"].isin([100.0, 200.0, 300.0])) # firmware codes
print(f"legacy PS-L rows: {legacy_mask.sum()} / {len(df)} ({legacy_mask.mean():.2%})")

X: (1567, 590) | defect rate: 6.64% (104 fail / 1463 pass)
missing cells: 4.54%
timestamp range: 2008-07-19 11:55:00 to 2008-10-17 06:07:00
legacy PS-L rows: 146 / 1567 (9.32%)


## 2 · Preprocessing & class rebalancing


In [ ]:
# FIX 1: Chronological split (first 60% train, next 20% val, last 20% test)
n = len(df)
n_train = int(0.60 * n)
n_val = int(test_size * n)

# Split indices chronologically
train_idx = slice(0, n_train)
val_idx = slice(n_train, n_train + n_val)
test_idx = slice(n_train + n_val, n)

print(f"split: train={n_train}, val={n_val}, test={n - n_train - n_val}")

# Extract legacy mask for each split
legacy_train = legacy_mask.iloc[train_idx].values
legacy_val = legacy_mask.iloc[val_idx].values
legacy_test = legacy_mask.iloc[test_idx].values

# FIX 3: Mask legacy sensor values BEFORE imputing (to not pollute statistics)
X_raw_copy = X_raw.copy()
X_raw_copy[legacy_mask, 0] = np.nan  # sensor_0
X_raw_copy[legacy_mask, 1] = np.nan  # sensor_1

# FIX 1: Fit imputer and scaler on TRAIN rows only
imputer = SimpleImputer(strategy="mean", keep_empty_features=True)
X_train_imp = imputer.fit_transform(X_raw_copy[train_idx])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)

# Transform val and test with fitted objects
X_val_imp = imputer.transform(X_raw_copy[val_idx])
X_val_scaled = scaler.transform(X_val_imp)

X_test_imp = imputer.transform(X_raw_copy[test_idx])
X_test_scaled = scaler.transform(X_test_imp)

# FIX 3: After scaling, set sensor_0 and sensor_1 to 0.0 for legacy rows (masked, not fabricated)
X_train_scaled[legacy_train, 0] = 0.0
X_train_scaled[legacy_train, 1] = 0.0
X_val_scaled[legacy_val, 0] = 0.0
X_val_scaled[legacy_val, 1] = 0.0
X_test_scaled[legacy_test, 0] = 0.0
X_test_scaled[legacy_test, 1] = 0.0

# FIX 3: Append one-hot columns for legacy firmware codes (NOT scaled)
def add_legacy_features(X, legacy_mask_df, sensor1_vals):
    # Create one-hot columns
    is_legacy = legacy_mask_df.astype(float).values
    fw_100 = (sensor1_vals == 100.0).astype(float)
    fw_200 = (sensor1_vals == 200.0).astype(float)
    fw_300 = (sensor1_vals == 300.0).astype(float)

    # Append to X
    return np.column_stack([X, is_legacy, fw_100, fw_200, fw_300])

# Get sensor_1 values for each split
sensor1_train = df.iloc[train_idx]["sensor_1"].values
sensor1_val = df.iloc[val_idx]["sensor_1"].values
sensor1_test = df.iloc[test_idx]["sensor_1"].values

X_train = add_legacy_features(X_train_scaled, legacy_mask.iloc[train_idx], sensor1_train)
X_val = add_legacy_features(X_val_scaled, legacy_mask.iloc[val_idx], sensor1_val)
X_test = add_legacy_features(X_test_scaled, legacy_mask.iloc[test_idx], sensor1_test)

y_train = y_raw[train_idx]
y_val = y_raw[val_idx]
y_test = y_raw[test_idx]

print(f"train {X_train.shape} | defect rate: {y_train.mean():.2%}")
print(f"val   {X_val.shape} | defect rate: {y_val.mean():.2%}")
print(f"test  {X_test.shape} | defect rate: {y_test.mean():.2%}")

# FIX 1: Handle imbalance with pos_weight instead of SMOTE
neg_train = (1 - y_train).sum()
pos_train = y_train.sum()
pos_weight = neg_train / pos_train
print(f"pos_weight (neg/pos on train): {pos_weight:.2f}")

split: train=940, val=313, test=314
train (940, 594) | defect rate: 8.09%
val   (313, 594) | defect rate: 3.51%
test  (314, 594) | defect rate: 5.41%
pos_weight (neg/pos on train): 11.37


## 3 · Tensors & loaders


In [ ]:
F = X_train.shape[1]                                     # 594 (590 sensors + 4 legacy features)

def make_loader(X, y, shuffle):
    # FIX 2: Remove unsqueeze(1) - inputs stay [B, F]
    Xt = torch.from_numpy(X).float()
    yt = torch.from_numpy(y).float()
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(TensorDataset(Xt, yt), batch_size=HP["batch_size"],
                      shuffle=shuffle, generator=g if shuffle else None, drop_last=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val, y_val, shuffle=False)
test_loader  = make_loader(X_test, y_test, shuffle=False)
print(f"train batches: {len(train_loader)} | val batches: {len(val_loader)} | test batches: {len(test_loader)}")

train batches: 14 | val batches: 5 | test batches: 5


## 4 · Model — supervised autoencoder


In [ ]:
class WaferSAE(nn.Module):
    def __init__(self, d_in: int, d_hid: int, d_lat: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_in),
        )
        self.head = nn.Linear(d_lat, 1)

    def forward(self, x):
        x = x.flatten(start_dim=1)                       # tolerate [B, F] or [B, C, F]
        z = self.encoder(x)
        return self.decoder(z), self.head(z).squeeze(-1), z

model = WaferSAE(F, HP["hidden"], HP["latent"]).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

parameters: 380,051


## 5 · Training


In [ ]:
opt   = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["epochs"])
# FIX 1: Use pos_weight to handle class imbalance instead of SMOTE
bce   = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(DEVICE))

history = {"total": [], "recon": [], "clf": []}

for epoch in range(1, HP["epochs"] + 1):
    model.train()
    acc = dict(total=0.0, recon=0.0, clf=0.0); n_batches = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)

        reconstructed, logits, _ = model(batch_x)

        # FIX 2: Assert shapes match before loss
        assert reconstructed.shape == batch_x.shape, f"recon {reconstructed.shape} != batch_x {batch_x.shape}"
        assert logits.shape == batch_y.shape, f"logits {logits.shape} != batch_y {batch_y.shape}"

        recon_loss = torch.mean((reconstructed - batch_x) ** 2)     # per-element MSE
        clf_loss   = bce(logits, batch_y)
        loss       = recon_loss + HP["lambda_clf"] * clf_loss

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()

        acc["total"] += loss.item(); acc["recon"] += recon_loss.item(); acc["clf"] += clf_loss.item()
        n_batches += 1

    sched.step()
    for k in acc:
        history[k].append(acc[k] / n_batches)
    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:02d}/{HP['epochs']} | total {history['total'][-1]:.4f} "
              f"| recon {history['recon'][-1]:.4f} | clf {history['clf'][-1]:.4f}")

epoch 01/40 | total 2.0724 | recon 0.7910 | clf 1.2813
epoch 05/40 | total 0.7403 | recon 0.7131 | clf 0.0271
epoch 10/40 | total 0.6231 | recon 0.6199 | clf 0.0031
epoch 15/40 | total 0.5696 | recon 0.5683 | clf 0.0013
epoch 20/40 | total 0.5289 | recon 0.5278 | clf 0.0011
epoch 25/40 | total 0.5046 | recon 0.5038 | clf 0.0008
epoch 30/40 | total 0.4873 | recon 0.4864 | clf 0.0008
epoch 35/40 | total 0.4820 | recon 0.4812 | clf 0.0008
epoch 40/40 | total 0.4810 | recon 0.4803 | clf 0.0008


## 6 · Evaluation


In [ ]:
# === VERIFICATION ===
print("\n=== Verification checks ===")

# Verify chronological split
train_ts = df.iloc[train_idx]["timestamp"]
test_ts = df.iloc[test_idx]["timestamp"]
print(f"max(train timestamp) < min(test timestamp): {train_ts.max() < test_ts.min()}")
print(f"train ts range: {train_ts.min()} to {train_ts.max()}")
print(f"test ts range:  {test_ts.min()} to {test_ts.max()}")

# Verify scaler was fit on train only (mean of scaled train should be ~0)
print(f"\nscaler.mean_ shape: {scaler.mean_.shape}")
print(f"scaler.mean_ first 5: {scaler.mean_[:5]}")
train_mean_scaled = X_train_scaled[:, :590].mean(axis=0)
print(f"mean of SCALed train rows (first 5 features): {train_mean_scaled[:5]}")
print(f"scaler fit on train only: {'YES (scaled mean ~0)' if np.allclose(train_mean_scaled[:5], 0, atol=0.1) else 'NO'}")

# Verify legacy masking
print(f"\nLegacy rows in train: {legacy_train.sum()}")
print(f"Legacy rows with non-zero sensor_0 after masking: {(X_train_scaled[legacy_train, 0] != 0.0).sum()}")
print(f"Legacy rows with non-zero sensor_1 after masking: {(X_train_scaled[legacy_train, 1] != 0.0).sum()}")

# Verify prevalence
print(f"\nPrevalence - train: {y_train.mean():.2%}, val: {y_val.mean():.2%}, test: {y_test.mean():.2%}")


=== Verification checks ===
max(train timestamp) < min(test timestamp): True
train ts range: 2008-07-19 11:55:00 to 2008-09-20 05:34:00
test ts range:  2008-10-02 20:54:00 to 2008-10-17 06:07:00

scaler.mean_ shape: (590,)
scaler.mean_ first 5: [3.00930567e+03 2.49487057e+03 2.19891672e+03 1.33491322e+03
 1.26322019e+00]
mean of SCALed train rows (first 5 features): [-4.50970230e-14  1.64455329e-14 -1.23546120e-14  1.53018260e-14
 9.01359365e-15]
scaler fit on train only: YES (scaled mean ~0)

Legacy rows in train: 79
Legacy rows with non-zero sensor_0 after masking: 0
Legacy rows with non-zero sensor_1 after masking: 0

Prevalence - train: 8.09%, val: 3.51%, test: 5.41%


In [ ]:
# FIX 4: Choose threshold on VALIDATION set (maximize F1 on precision-recall curve)
model.eval()
val_probs = []
with torch.no_grad():
    for xb, _ in val_loader:
        _, logits, _ = model(xb.to(DEVICE))
        val_probs.append(torch.sigmoid(logits).cpu())
val_probs = torch.cat(val_probs).numpy()

# Find threshold that maximizes F1 on validation set
from sklearn.metrics import precision_recall_curve
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
print(f"Best threshold on validation set: {best_threshold:.4f} (F1={f1_scores[best_idx]:.4f})")

# FIX 4: Evaluate once on untouched test set
test_probs = []
with torch.no_grad():
    for xb, _ in test_loader:
        _, logits, _ = model(xb.to(DEVICE))
        test_probs.append(torch.sigmoid(logits).cpu())
test_probs = torch.cat(test_probs).numpy()
test_preds = (test_probs >= best_threshold).astype(int)

# Compute metrics
f1   = f1_score(y_test, test_preds)
prec = precision_score(y_test, test_preds)
rec  = recall_score(y_test, test_preds)
auc  = roc_auc_score(y_test, test_probs)
ap   = average_precision_score(y_test, test_probs)
tn, fp, fn, tp = confusion_matrix(y_test, test_preds).ravel()

print(f"\n=== Test set evaluation (natural prevalence) ===")
print(f"test size: {len(y_test)}")
print(f"test defect prevalence: {y_test.mean():.2%}")
print(f"threshold: {best_threshold:.4f}")
print(f"F1 {f1:.4f} | precision {prec:.4f} | recall {rec:.4f}")
print(f"ROC-AUC {auc:.4f} | PR-AUC (AP) {ap:.4f}")
print(f"confusion  TN={tn}  FP={fp}  FN={fn}  TP={tp}")


Best threshold on validation set: 0.0000 (F1=0.0796)

=== Test set evaluation (natural prevalence) ===
test size: 314
test defect prevalence: 5.41%
threshold: 0.0000
F1 0.1272 | precision 0.0705 | recall 0.6471
ROC-AUC 0.6393 | PR-AUC (AP) 0.0863
confusion  TN=152  FP=145  FN=6  TP=11



## 7 · Release gate



In [ ]:
# FIX 4: Change release gate to only print measured F1 (remove PASSED logic)
print(f"Measured F1 on test set (natural prevalence): {f1:.4f}")

os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/wafer_sae.pt")
joblib.dump({"imputer": imputer, "scaler": scaler, "threshold": best_threshold}, "artifacts/preprocessor.joblib")
print("Artifacts written to ./artifacts/  — ready for the deployment pipeline.")


Measured F1 on test set (natural prevalence): 0.1272
Artifacts written to ./artifacts/  — ready for the deployment pipeline.
